# Food Security Analyst Assistant — RAG Baseline

A minimal RAG pipeline over real Somalia food security reports (IPC acute food insecurity analyses, FSNAU reports, WFP situation reports/country briefs — see `data/SOURCES.md`), following the same structure used in the course: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

In [1]:
import json
import re
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from pypdf import PdfReader
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Load the real Somalia IPC/FSNAU/WFP reports in `data/` (see `data/SOURCES.md` for the full list and provenance), extract text from each PDF, split it into overlapping chunks, and attach metadata to every chunk: source organisation, report title, publication date, and region (when a single Somalia region is mentioned in that chunk).

In [2]:
DATA_DIR = Path("../data")

# Organisation, title and publication date for each report. See data/SOURCES.md
# for URLs and notes on provenance (including the Post Deyr date discrepancy).
REPORTS = {
    "ipc_somalia_post_deyr_2025_findings_feb2026.pdf": {
        "source_org": "IPC",
        "title": "Somalia 2025/26 Post Deyr IPC Analysis - Briefing Presentation for All Stakeholders",
        "publication_date": "2026-02-24",
    },
    "ipc_somalia_apr_jun2025_report.pdf": {
        "source_org": "IPC",
        "title": "Somalia IPC Acute Food Insecurity and Acute Malnutrition Analysis - Updated Projection, April-June 2025",
        "publication_date": "2025-03-29",
    },
    "ipc_somalia_jul_dec2026_report.pdf": {
        "source_org": "IPC",
        "title": "Somalia IPC Acute Food Insecurity and Acute Malnutrition Analysis, July-December 2026 (Risk of Famine)",
        "publication_date": "2026-09-23",
    },
    "ipc_somalia_jul_dec2026_snapshot.pdf": {
        "source_org": "IPC",
        "title": "Somalia IPC Acute Food Insecurity and Malnutrition Snapshot, July-December 2026",
        "publication_date": "2026-09-23",
    },
    "fsnau_post_gu_2026_nutrition_summary.pdf": {
        "source_org": "FSNAU",
        "title": "Somalia 2026 Post Gu Key Nutrition Results Summary",
        "publication_date": "2026-09-23",
    },
    "fsnau_cfsva_2026.pdf": {
        "source_org": "FSNAU",
        "title": "Somalia Comprehensive Food Security and Vulnerability Analysis (CFSVA) 2026",
        "publication_date": "2026",
    },
    "wfp_somalia_country_brief_feb2026.pdf": {
        "source_org": "WFP",
        "title": "WFP Somalia Country Brief, February 2026",
        "publication_date": "2026-02",
    },
    "wfp_somalia_country_brief_nov2025.pdf": {
        "source_org": "WFP",
        "title": "WFP Somalia Country Brief, November 2025",
        "publication_date": "2025-11",
    },
}

SOMALIA_REGIONS = [
    "Awdal", "Woqooyi Galbeed", "Togdheer", "Sanaag", "Sool", "Bari", "Nugaal",
    "Mudug", "Galgaduud", "Hiraan", "Middle Shabelle", "Banadir", "Lower Shabelle",
    "Bay", "Bakool", "Gedo", "Middle Juba", "Lower Juba",
]


def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def detect_region(text):
    found = {region for region in SOMALIA_REGIONS if re.search(rf"\b{re.escape(region)}\b", text)}
    if len(found) == 1:
        return found.pop()
    if len(found) > 1:
        return "multiple"
    return None


def sliding_window(text, size=1500, step=750):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunks.append(text[start:end])
        start = end - step
        if end >= text_length:
            break

    return chunks


documents = []

for filename, meta in REPORTS.items():
    full_text = extract_text(DATA_DIR / filename)
    for chunk_id, chunk_text in enumerate(sliding_window(full_text)):
        documents.append({
            "filename": filename,
            "source_org": meta["source_org"],
            "title": meta["title"],
            "publication_date": meta["publication_date"],
            "region": detect_region(chunk_text),
            "chunk_id": chunk_id,
            "content": chunk_text,
        })

len(documents)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/RHLWSP+MyriadPro-Regular', '/Encoding': IndirectObject(4023, 0, 4825075856), '/FirstChar': 30, '/FontDescriptor': IndirectObject(4024, 0, 4825075856), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [596, 523, 212, 0, 0, 0, 0, 792, 0, 0, 284, 284, 0, 0, 0, 307, 0, 0, 513, 0, 513, 0, 513, 513, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 612, 0, 580, 666, 492, 487, 0, 0, 239, 0, 0, 0, 0, 0, 0, 532, 0, 538, 493, 0, 647, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 482, 569, 448, 564, 501, 292, 559, 555, 234, 0, 469, 236, 834, 555, 549, 0, 563, 327, 396, 331, 551, 481, 736, 0, 471]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FGVRGN+MyriadPro-Semibold', '/Encoding': IndirectObject(4025, 0, 4825075856), '/FirstChar': 31, '/FontDescriptor': IndirectObject(4020, 0, 4825075856), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [559, 207, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 636, 0, 588, 0, 0, 509, 0, 0, 264, 0, 582, 0, 827, 0, 0, 559, 0, 0, 519, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 508, 585, 449, 581, 516, 319, 0, 572, 256, 0, 0, 257, 848, 572, 564, 585, 0, 356, 417, 351, 569, 0, 0, 0, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/TESSUL+MyriadPro-Light', '/Encoding': IndirectObject(4026, 0, 4825075856), '/FirstChar': 31, '/FontDescriptor': IndirectObject(4015, 0, 4825075856), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [542, 219, 0, 0, 0, 0, 737, 0, 0, 265, 265, 0, 0, 0, 0, 0, 0, 488, 0, 488, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 502, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 454, 0, 446, 544, 485, 262, 544, 536, 209, 0, 0, 212, 819, 536, 532, 552, 0, 295, 373, 309, 531, 452, 0, 0, 440]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/RHLWSP+MyriadPro-Regular', '/Encoding': IndirectObject(4023, 0, 4825075856), '/FirstChar': 30, '/FontDescriptor': IndirectObject(4024, 0, 4825075856), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [596, 523, 212, 0, 0, 0, 0, 792, 0, 0, 284, 284, 0, 0, 0, 307, 0, 0, 513, 0, 513, 0, 513, 513, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 612, 0, 580, 666, 492, 487, 0, 0, 239, 0, 0, 0, 0, 0, 0, 532, 0, 538, 493, 0, 647, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 482, 569, 448, 564, 501, 292, 559, 555, 234, 0, 469, 236, 834, 555, 549, 0, 563, 327, 396, 331, 551, 481, 736, 0, 471]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FGVRGN+MyriadPro-Semibold', '/Encoding': IndirectObject(4025, 0, 4825075856), '/FirstChar': 31, '/FontDescriptor': IndirectObject(4020, 0, 4825075856), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [559, 207, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 636, 0, 588, 0, 0, 509, 0, 0, 264, 0, 582, 0, 827, 0, 0, 559, 0, 0, 519, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 508, 585, 449, 581, 516, 319, 0, 572, 256, 0, 0, 257, 848, 572, 564, 585, 0, 356, 417, 351, 569, 0, 0, 0, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


...

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/GUMVLH+MyriadPro-Light', '/Encoding': IndirectObject(419, 0, 4824780736), '/FirstChar': 31, '/FontDescriptor': IndirectObject(415, 0, 4824780736), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [542, 219, 0, 0, 0, 0, 737, 0, 0, 265, 265, 0, 0, 0, 0, 0, 0, 488, 0, 488, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 502, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 454, 0, 446, 544, 485, 262, 544, 536, 209, 0, 0, 212, 819, 536, 532, 552, 0, 295, 373, 309, 531, 452, 0, 0, 440]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JFGORB+MyriadPro-Regular', '/Encoding': IndirectObject(417, 0, 4824780736), '/FirstChar': 30, '/FontDescriptor': IndirectObject(422, 0, 4824780736), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [596, 523, 212, 0, 0, 0, 0, 792, 0, 0, 284, 284, 0, 0, 0, 307, 0, 0, 513, 0, 513, 0, 513, 513, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 612, 0, 580, 666, 492, 487, 0, 0, 239, 0, 0, 0, 0, 0, 0, 532, 0, 538, 493, 0, 647, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 482, 569, 448, 564, 501, 292, 559, 555, 234, 0, 469, 236, 834, 555, 549, 0, 563, 327, 396, 331, 551, 481, 736, 0, 471]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JFGORB+MyriadPro-Semibold', '/Encoding': IndirectObject(434, 0, 4824780736), '/FirstChar': 31, '/FontDescriptor': IndirectObject(421, 0, 4824780736), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [559, 207, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 636, 0, 588, 0, 0, 509, 0, 0, 264, 0, 582, 0, 827, 0, 0, 559, 0, 0, 519, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 508, 585, 449, 581, 516, 319, 0, 572, 256, 0, 0, 257, 848, 572, 564, 585, 0, 356, 417, 351, 569, 0, 0, 0, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/EYEKEV+MyriadPro-Regular', '/Encoding': IndirectObject(934, 0, 4827726480), '/FirstChar': 31, '/FontDescriptor': IndirectObject(1771, 0, 4827726480), '/LastChar': 119, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [523, 212, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 513, 0, 513, 513, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 612, 0, 0, 666, 492, 487, 646, 0, 239, 0, 542, 0, 804, 658, 0, 0, 0, 538, 493, 497, 0, 0, 0, 0, 541, 0, 0, 0, 0, 0, 0, 0, 482, 0, 0, 564, 501, 292, 559, 555, 234, 0, 0, 0, 834, 555, 549, 0, 0, 327, 396, 331, 0, 0, 736]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


509

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over `title` and `content`, plus exact-match keyword fields for `source_org`, `region`, and `filename` so results can be traced back to a specific report.

In [3]:
index = Index(
    text_fields=["title", "content"],
    keyword_fields=["source_org", "region", "filename"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant report chunks, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three. `instructions` tells the model to cite sources, separate stated causes from mere correlation, and flag thin or missing evidence.

In [4]:
def search(query):
    return index.search(query, num_results=8)

In [5]:
instructions = """
You are a food security analyst assistant. Answer the QUESTION using only
the passages in CONTEXT, which are chunks retrieved from IPC acute food
insecurity analyses, FSNAU reports, and WFP situation reports/country briefs
on Somalia.

Rules:
- Cite the source report and its publication date for every factual claim,
  using the "title" and "publication_date" fields from the matching context
  chunk, e.g. "(WFP Somalia Country Brief, February 2026)".
- Only state a cause of food insecurity (e.g. conflict, drought, poor
  rainfall, funding cuts) if a retrieved passage explicitly states that
  cause. If the context only shows that two things changed around the same
  time without a source stating a causal link, describe it as a correlation
  or co-occurrence, not a cause, and say so explicitly.
- If the retrieved context does not cover the question (e.g. no chunks
  mention the requested region, season, or driver), say plainly that the
  available evidence does not cover it. Do not fill the gap with general
  knowledge or speculation.
- If the evidence is thin (only one source, or chunks that mention the topic
  only in passing) or stale (the most relevant chunk is from an older
  season than the question asks about), say so explicitly rather than
  presenting it with the same confidence as strong, recent evidence.
- Never state or imply a causal link that the retrieved text does not
  support.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

Ask the question this project is built around, and show the retrieved chunks (with their sources) alongside the answer.

In [8]:
query = "What is driving food insecurity in Bay region this season, and have conflict events or rainfall changed there?"

retrieved = search(query)

print(f"Retrieved {len(retrieved)} chunks:\n")
for r in retrieved:
    print(f"- [{r['source_org']}] {r['title']} ({r['publication_date']}) | region={r['region']} | chunk {r['chunk_id']} of {r['filename']}")

print("\n--- Answer ---\n")
print(rag(query))

Retrieved 8 chunks:

- [IPC] Somalia IPC Acute Food Insecurity and Malnutrition Snapshot, July-December 2026 (2026-09-23) | region=None | chunk 0 of ipc_somalia_jul_dec2026_snapshot.pdf
- [IPC] Somalia IPC Acute Food Insecurity and Malnutrition Snapshot, July-December 2026 (2026-09-23) | region=None | chunk 18 of ipc_somalia_jul_dec2026_snapshot.pdf
- [IPC] Somalia IPC Acute Food Insecurity and Acute Malnutrition Analysis, July-December 2026 (Risk of Famine) (2026-09-23) | region=None | chunk 2 of ipc_somalia_jul_dec2026_report.pdf
- [IPC] Somalia IPC Acute Food Insecurity and Acute Malnutrition Analysis, July-December 2026 (Risk of Famine) (2026-09-23) | region=None | chunk 3 of ipc_somalia_jul_dec2026_report.pdf
- [IPC] Somalia IPC Acute Food Insecurity and Malnutrition Snapshot, July-December 2026 (2026-09-23) | region=None | chunk 19 of ipc_somalia_jul_dec2026_snapshot.pdf
- [IPC] Somalia IPC Acute Food Insecurity and Acute Malnutrition Analysis, July-December 2026 (Risk of Famine)

In the Bay region this season, food insecurity is primarily driven by the ongoing impacts of drought, conflict, and high food prices. The erratic and below-average rainfall during the April-to-June 2026 Gu season significantly affected agropastoral areas, persisting from previous drought conditions in mid-to-late 2025 (Somalia IPC Acute Food Insecurity and Malnutrition Snapshot, July-December 2026, September 2026). 

Additionally, conflict and insecurity continue to displace populations, disrupt market access, and constrain livelihoods and humanitarian assistance (Somalia IPC Acute Food Insecurity and Acute Malnutrition Analysis, July-December 2026 (Risk of Famine), September 2026). Flooding from El Niño is also expected to exacerbate conditions, with nearly 26,000 internally displaced persons in Baidoa facing a risk of famine (Somalia IPC Acute Food Insecurity and Malnutrition Snapshot, July-December 2026, September 2026).

Regarding rainfall changes, the Gu rains were better than pre